## Session 2 · Topic 8 — Cleaning: dtypes, missing values and simple fixes (Python)

**Dataset:** `Y1_T1_2025.xlsx` — Term 1, 2025 first-year unit-attempt extract
(one row per student unit attempt).

In the theory topic you saw the two problems that distort almost every analysis:
**missing values** and **duplicate rows**, plus a related trap — **wrong data
types**. This notebook practises all three on the real extract, so it is clean
and trustworthy before you summarise it in Session 3.

You will:

1. Check the column **dtypes** and spot what looks wrong.
2. **Fix a wrong type** — a date stored as text.
3. **Find and handle missing values** sensibly.
4. Apply a **simple fix** — remove duplicate rows.

Replace every `# TODO` with your own code and run the cell.

### 1. Load the data

Read the **`Extract`** sheet of `data/Y1_T1_2025.xlsx` into `df` and print its
shape.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("../data")
DATA_FILE = DATA_FOLDER / "Y1_T1_2025.xlsx"

df = pd.read_excel(DATA_FILE, sheet_name="Extract")
print(df.shape)

### 2. Check the data types

Every column has a **dtype** — how pandas stores it. `object` usually means text,
`int64`/`float64` are numbers. The wrong type quietly breaks calculations, so
check first.

Print `df.dtypes` and look for anything surprising.

In [ ]:
df.dtypes

Two things worth noticing:

- **`mark` is `float64`, even though marks are whole numbers (0–100).** That is
  not an error — a column with any missing value *cannot* be a plain integer,
  because there is no whole number for "missing", so pandas uses `float` to hold
  `NaN`. The float is a signal that data is missing.
- **`date_of_birth` is `object` (text), not a date.** That is the wrong type —
  you can't do date arithmetic on text. You'll fix it next.

Confirm the second point: print `df["date_of_birth"].head(3)` and check
`type(df["date_of_birth"].iloc[0])` — it's a Python `str`.

In [ ]:
print(df["date_of_birth"].head(3).tolist())
print(type(df["date_of_birth"].iloc[0]))

### 3. Fix the wrong type

Convert `date_of_birth` from text to a real date with `pd.to_datetime()`, writing
it back to the same column. Then print `df["date_of_birth"].dtype` — it should now
be `datetime64[ns]`.

Because it is a real date now, you can pull parts out of it. Print
`df["date_of_birth"].dt.year.head()` to prove it works.

In [ ]:
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"])
print(df["date_of_birth"].dtype)
print(df["date_of_birth"].dt.year.head())

### 4. Find the missing values

`df.isna().sum()` counts missing values in every column. Print it and note which
columns have gaps.

You should find two: `phone` (some students never gave a number) and `mark`.

In [ ]:
df.isna().sum()

### 5. Handle the missing marks — think before you fill

It is tempting to fill missing marks with the average. **Don't.** A missing
`mark` here means the student **withdrew** and never sat the unit — they have no
result. Inventing a mark for them would distort every average and pass rate.

Confirm that: look at `unit_attempt_status` for the rows where `mark` is missing
— they should all be `Withdrawn`.

In [ ]:
df_missing_mark = df[df["mark"].isna()]
print(df_missing_mark["unit_attempt_status"].value_counts())

So the right move depends on the question:

- For anything about **marks** (average mark, pass rate), **drop** the rows with
  no mark — they aren't part of that question.
- For anything about **enrolments or withdrawals**, **keep** them — the missing
  mark is itself the information.

Build a marks-only table for the first kind of question: drop rows where `mark`
is missing into `marks_df`, and print how many rows remain.

In [ ]:
df_marks = df.dropna(subset=["mark"])
print(len(df), "->", len(df_marks))

### 6. A simple fix — remove duplicate rows

This extract has a quirk: a student with two phone numbers on file appears
**twice**, once per number. Those rows describe the *same unit attempt*, so they
are duplicates for any analysis that ignores the phone.

`df.duplicated().sum()` counts only rows that are identical in **every** column —
here that's `0`, because the phone differs. The fix is to define what makes an
attempt unique and dedup on just those **key** columns with `subset=`:

`student_id`, `unit_code`, `availability_year`, `study_period`, `attempt_no`.

Count exact duplicates first (to show it misses them), then `drop_duplicates()`
on the key columns and print the before/after row counts.

In [ ]:
print("exact duplicate rows:", df.duplicated().sum())

key = ["student_id", "unit_code", "availability_year", "study_period", "attempt_no"]
df_attempts = df.drop_duplicates(subset=key)
print(len(df), "->", len(df_attempts))

### 7. Wrap-up

In your own words (2–3 sentences): which column had the wrong type and how did you
fix it; why was filling the missing marks the *wrong* choice here; and why did
`duplicated()` on its own miss the duplicate rows?